# Project FORESIGHT — 03 Machine Learning Forecasting Model

This notebook trains a scikit-learn `HistGradientBoostingRegressor` on the engineered sales features and evaluates its performance via rolling-origin backtesting against the Seasonal-Naive baseline.

In [ ]:
import os
import pickle
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import HistGradientBoostingRegressor

%matplotlib inline
sns.set_theme(style="whitegrid")

## 1. Load Processed Dataset

In [ ]:
df = pd.read_csv("../data/processed/clean_sales.csv")
df["date_dt"] = pd.to_datetime(df["date"])
print(f"Loaded dataset: {df.shape[0]} rows across {df['sku_id'].nunique()} SKUs")

## 2. Model Evaluation (Rolling-Origin Cross-Validation)

We evaluate our model using three 6-week rolling validation folds to ensure no future data leakage.

In [ ]:
# Define evaluation helper functions
def calculate_wape(actual, forecast):
    return np.sum(np.abs(actual - forecast)) / np.sum(actual)

def calculate_bias(actual, forecast):
    return np.sum(forecast - actual) / np.sum(actual)

def prepare_ml_features(data, is_train=True):
    feature_cols = [
        "promo_flag", "is_holiday", "day_of_week", "day_of_month", "quarter", "month",
        "units_sold_lag_1", "units_sold_lag_7", "units_sold_lag_28",
        "units_sold_roll_mean_7", "units_sold_roll_std_7",
        "units_sold_roll_mean_28", "units_sold_roll_std_28"
    ]
    categorical_cols = ["category", "subcategory", "season", "promo_event"]
    
    data_enc = data.copy()
    for col in categorical_cols:
        if col in data_enc.columns:
            data_enc[col] = data_enc[col].astype("category")
            
    all_features = feature_cols + [c for c in categorical_cols if c in data_enc.columns]
    if is_train:
        return data_enc[all_features], data_enc["units_sold"]
    else:
        return data_enc[all_features]

In [ ]:
# Execute evaluation across folds
unique_dates = sorted(df["date_dt"].unique())
n_days = len(unique_dates)
fold_horizon = 42

folds = [
    (n_days - 3 * fold_horizon, n_days - 2 * fold_horizon),
    (n_days - 2 * fold_horizon, n_days - fold_horizon),
    (n_days - fold_horizon, n_days)
]

for i, (train_cut, test_end) in enumerate(folds):
    train_date = unique_dates[train_cut]
    test_date = unique_dates[test_end - 1]
    
    train_df = df[df["date_dt"] < train_date]
    test_df = df[(df["date_dt"] >= train_date) & (df["date_dt"] <= test_date)]
    
    X_tr, y_tr = prepare_ml_features(train_df)
    X_te, y_te = prepare_ml_features(test_df)
    
    cat_indices = [X_tr.columns.get_loc(c) for c in ["category", "subcategory", "season", "promo_event"]]
    
    model = HistGradientBoostingRegressor(categorical_features=cat_indices, random_state=42)
    model.fit(X_tr, y_tr)
    
    preds = np.clip(model.predict(X_te), 0.0, None)
    wape = calculate_wape(y_te.values, preds)
    bias = calculate_bias(y_te.values, preds)
    
    print(f"Fold {i+1} Test metrics -> WAPE: {wape:.4f}, Bias: {bias:.4f}")

## 3. Train Final Model and Save
Train model on the entire historical period to predict the future.

In [ ]:
X_final, y_final = prepare_ml_features(df)
cat_indices = [X_final.columns.get_loc(c) for c in ["category", "subcategory", "season", "promo_event"]]

final_model = HistGradientBoostingRegressor(categorical_features=cat_indices, random_state=42)
final_model.fit(X_final, y_final)

# Show prediction intervals distribution
residuals = y_final - final_model.predict(X_final)
std_res = np.std(residuals)
print(f"Standard deviation of training residuals: {std_res:.4f}")

plt.figure(figsize=(8, 4))
sns.histplot(residuals, kde=True, bins=50, color="purple")
plt.title("Histogram of Training Residuals")
plt.xlabel("Residual (Actual - Predicted)")
plt.ylabel("Count")
plt.show()